# interpolated data from trimmed raw data 

In [1]:
import os
import glob
import numpy as np
import pandas as pd


#Save interpolated data csv

## Find trimmed files

In [2]:
#Input 
INPUT_DIR  = r"./../../dataFolders/MuscaChasingBeads/xyz_Raw_Trimmed/"
OUTPUT_DIR = r"./../../dataFolders/MuscaChasingBeads/xyz_Interpolated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_files = glob.glob(os.path.join(INPUT_DIR, "*_trimmed.csv"))
if not csv_files:
    raise FileNotFoundError("No trimmed CSVs found")

In [3]:
def intepol_limit_for_ind_pose(df):
    
    points_for_shorter_interpol = ['pt5', 'pt6']
    short_column_list  = [cols for cols in df.columns if 
                   any(k in cols for k in points_for_shorter_interpol)]
    short_gap = [7] * len(short_column_list)

    points_for_longer_interpol = ['pt1', 'pt2', 'pt3', 'pt4', 'center', 'pt7']
    long_column_list = [cols for cols in df.columns if 
                   any(k in cols for k in points_for_longer_interpol)]
    long_gap = [20] * len(long_column_list)

    columns = short_column_list + long_column_list
    gaps = short_gap + long_gap
    column_limits = dict(zip(columns, gaps))
    
    return column_limits

## run the code across all files

In [4]:
# Interpolate and save
for path in csv_files:
    fname = os.path.basename(path)
    print("Interpolating:", fname)

    df = pd.read_csv(path)
    
    # ---- interpolate all points - cubic ----
    interp_df = df.copy()

    column_limits = intepol_limit_for_ind_pose(df)
    
    # Apply the interpolation to each column with its specified limit
    for col, gaplength in column_limits.items():
        interp_df[col] = df[col].interpolate(method = 'cubicspline'
                                             ,limit = gaplength
                                            ,limit_area = 'inside')
        
    # --------------------------------------------------------------
    
    out_name = fname.replace("_trimmed.csv", "_INTERP.csv")
    interp_df.to_csv(os.path.join(OUTPUT_DIR, out_name), index=False)

print("Interpolated CSVs saved.")

Interpolating: Trial2_180rpmxyzpts_trimmed.csv
Interpolating: Trial2_200rpmxyzpts_trimmed.csv
Interpolating: Trial3_180rpmxyzpts_trimmed.csv
Interpolating: Trial4_400rpmxyzpts_trimmed.csv
Interpolating: Trial5_180rpmxyzpts_trimmed.csv
Interpolating: Trial5_400rpmxyzpts_trimmed.csv
Interpolating: Trial7_400rpmxyzpts_trimmed.csv
Interpolated CSVs saved.


In [5]:
column_limits

{'pt5_X': 7,
 'pt5_Y': 7,
 'pt5_Z': 7,
 'pt6_X': 7,
 'pt6_Y': 7,
 'pt6_Z': 7,
 'pt1_X': 20,
 'pt1_Y': 20,
 'pt1_Z': 20,
 'pt2_X': 20,
 'pt2_Y': 20,
 'pt2_Z': 20,
 'pt3_X': 20,
 'pt3_Y': 20,
 'pt3_Z': 20,
 'pt4_X': 20,
 'pt4_Y': 20,
 'pt4_Z': 20,
 'pt7_X': 20,
 'pt7_Y': 20,
 'pt7_Z': 20,
 'center_X': 20,
 'center_Y': 20,
 'center_Z': 20}